[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dcintlab/WignerCamp2026/blob/master/AnomalyDetectionALTx/altx_anomaly.ipynb)

# `ALTx` for anomaly detection

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive/')
    %cd /content/drive/My Drive/Colab Notebooks/WignerCamp2025
except:
    IN_COLAB = False
    %load_ext autoreload
    %autoreload 2
print(f'Running on {"Google colab" if IN_COLAB else "Local computer"}')

Running on Local computer


In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib as mpl

from functions import *

## Step 0: Define a training and test set
We generate synthetic time series using simple **linear recursion rules**:
- Class `a` is generated using the Fibonacci rule: $x_n = x_{n-1} + x_{n-2}$. ($w=[1, 1]$).
- Class `b` uses a different rule: $x_n = 2x_{n-2} + x_{n-1}$. ($w=[2, 1]$).
- The test sample also follows the Fibonacci rule but starts from different initial values.

In [4]:
a = generate_recursive_array([1, 1], [1, 1], 5)
print(f"Train instance for class 'a': {a}")
b = generate_recursive_array([2, 1], [2, 1], 5)
print(f"Train instance for class 'b': {b}")
x = generate_recursive_array([1, 1], [2, 3], 5)
print(f"Test instance 'x': {x} belonging to an unknown class.")

Train instance for class 'a': tensor([1., 1., 2., 3., 5.])
Train instance for class 'b': tensor([ 2.,  1.,  5.,  7., 17.])
Test instance 'x': tensor([ 2.,  3.,  5.,  8., 13.]) belonging to an unknown class.


## Step 1: Extracting the Laws
We extract **shapelet vectors** (or 'laws') from each training instance. Each law is a direction in space that characterizes how values change locally.

We use parameters $R=3$, $L=2$, $K=1$, meaning:
- $R=3$: the length of subsequences considered.
- $L=2$: the size of the matrix constructed from those sequences.
- $K=1$: the shift between consecutive windows.

The shapelet vectors are obtained by computing the eigenvector of a symmetric matrix (constructed via time-delay embedding) corresponding to the **smallest eigenvalue**.

In [5]:
laws_a = extract_symmetric_laws(a)
laws_b = extract_symmetric_laws(b)
print(f"Laws for class 'a':\n{laws_a}")
print(f"Laws for class 'b':\n{laws_b}")

Laws for class 'a':
tensor([[-0.8507, -0.8507, -0.8507],
        [ 0.5257,  0.5257,  0.5257]])
Laws for class 'b':
tensor([[-0.9571, -0.8702, -0.9085],
        [ 0.2898,  0.4927,  0.4179]])


## Step 2: Embedding the test instance
We now embed the test instance into 2D vectors (pairs of values) that will be multiplied by the laws. This step constructs an input matrix $E$.

In [6]:
embedded = embed_as_pairs(x)
norm = torch.norm(embedded, dim=1)
print(f"The embedded test instance: \n{embedded}")
embedded = embedded / norm.unsqueeze(1)

The embedded test instance: 
tensor([[ 2.,  3.],
        [ 3.,  5.],
        [ 5.,  8.],
        [ 8., 13.]])


## Step 3: Projection by Laws (Matrix Multiplication)
We apply the transformation by multiplying the embedded matrix with the laws of each class. The goal is that the *correct laws* will make the result close to zero vectors, while *incorrect laws* won’t.

In [7]:
F_a = embedded @ laws_a
F_b = embedded @ laws_b
print(f"Multiplying with laws from class 'a' gives:\n{F_a}")
print(f"Multiplying with laws from class 'b' gives:\n{F_b}")

Multiplying with laws from class 'a' gives:
tensor([[-0.0344, -0.0344, -0.0344],
        [ 0.0132,  0.0132,  0.0132],
        [-0.0050, -0.0050, -0.0050],
        [ 0.0019,  0.0019,  0.0019]])
Multiplying with laws from class 'b' gives:
tensor([[-0.2898, -0.0727, -0.1563],
        [-0.2439, -0.0252, -0.1091],
        [-0.2615, -0.0434, -0.1272],
        [-0.2548, -0.0365, -0.1203]])


## Step 4: Feature Calulation
From the result of the transformation, we compute a simple **statistical feature**: the average of the squared values (mean of $F^2$). Lower values indicate better alignment with the laws of a class.

In [8]:
res_a = torch.mean(F_a**2)
res_b = torch.mean(F_b**2)
print(f"For class 'a': mean_all: {res_a}")
print(f"For class 'b': mean_all: {res_b}")
print(f"\nBased on the method the unknown instance belongs to class: '{'a' if res_a < res_b else 'b'}'.")

For class 'a': mean_all: 0.00034670191234909
For class 'b': mean_all: 0.029409073293209076

Based on the method the unknown instance belongs to class: 'a'.


## Summary
In this minimal example, ALT successfully classified the test sequence. The method:
- Extracted characteristic patterns (laws) from the training classes.
- Transformed the new sequence into a comparison-friendly space.
- Used simple statistics to determine which class's laws best describe the new data.

This illustrates how **ALT builds an interpretable and efficient classification pipeline**, leveraging linear algebra and time-delay embeddings.

For more advanced applications, ALT supports multiple channels, longer sequences, and sophisticated feature extraction techniques like percentiles and higher moments.